# k-means Clustering

## Table of contents

[1.) Simple k-means clustering example](#section_1)<br>
[2.) Image segmentation](#section_2)<br>
[3.) 3D point cloud segmentation](#section_3)<br>
[4.) Finding clusters in the apartment data](#section_4)

## Libraries and settings

In [ ]:
# Install opencv-python-headless (neccessary in GitHub Codespaces)
!pip install --upgrade pip
!pip install opencv-python-headless

# Libraries
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Current working directory
print('\nCurrent working directory:', os.getcwd())

## 1.) Simple k-means clustering example
<a id='section_1'></a>

### Create the dataset

In [ ]:
# Create data
centers = [[2,1], [-2,2], [-2,-2], [-4,-5], [5,7]]
X, y = make_blobs(n_samples=300, 
                  centers=centers, 
                  cluster_std=0.8,
                  random_state=42)

# Normalization of the values
X = StandardScaler().fit_transform(X)

# Plot the data
plt.figure(figsize=(6,4))
plt.scatter(X[:,0], X[:,1], s=10, color='darkred')
plt.show()

### Elbow Method showing the optimal k

In [ ]:
# Sum of squared distances of samples to their closest cluster center
distortions = []

# Range of k's
K = range(1,16,1)

# Loop to find the optimal k
for k in K:
    kmeanModel = KMeans(n_clusters=k)
    kmeanModel.fit(X)
    distortions.append(kmeanModel.inertia_)
    
# Elbow plot
plt.figure(figsize=(5,3))
plt.plot(K, distortions, 'bx-')
plt.xlabel('k')
plt.ylabel('Distortion')
plt.title('The Elbow Method showing the optimal k')

plt.show()

### Task 1b: Optimal k from Elbow Chart

Based on the elbow chart above, the optimal value of k appears to be **5**. This is where the "elbow" or bend in the curve occurs - the point where the rate of decrease in distortion sharply slows down. Beyond k=5, adding more clusters provides diminishing returns in reducing the within-cluster sum of squares.

### Perform k-means clustering

In [ ]:
# Number of clusters
k = 5

# k-means clustering
kmeans = KMeans(n_clusters=k, random_state=42).fit(X)

# Predict the values
y2 = kmeans.predict(X)

# Plot the clusters
plt.figure(figsize=(6,4))
plt.scatter(X[:, 0], X[:, 1], c=y2, s=10)
plt.show()

### Get and check the converged cluster centroids

In [ ]:
# Print centroids
centroids = kmeans.cluster_centers_
print('Cluster centroids:')
print(centroids, '\n')

# Check the 1st cluster's centroid 'by hand'
clust_00 = X[y2 == 0]
print('The 1st cluster\'s centroid:')
print(f'x = {sum(clust_00[:,0]/len(clust_00[:,0])):.8f}')
print(f'y = {sum(clust_00[:,1]/len(clust_00[:,1])):.8f}')

### Get the inertia or 'within-cluster sum-of-squares (WCSS)' of the k-means model

In [ ]:
print(f'Within-cluster sum-of-squares: {kmeans.inertia_:.4f}')

### Perform Silhouette Analysis - Task 1c: Extended for k = 2 to 11
- For examples see:
- https://laid-back-scientist.com/en/k-means
- https://machinelearninggeek.com/evaluating-clustering-methods
- https://medium.com/@favourphilic/how-to-interpret-silhouette-plot-for-k-means-clustering-414e144a17fe

In [ ]:
# Import own module for Silhouette plots
from silhouette import *

# Create Silhouette plots for different k's (Task 1c: Extended to k=2 through k=11)
# Note: range(2,12,1) provides: [2, 3, 4, 5, 6, 7, 8, 9, 10, 11]
for i in range(2,12,1):
    model = KMeans(n_clusters=i, 
                   random_state=42, 
                   init='random')
    model.fit(X)
    print(f'k={i}, Silhouette Score: {silhouette_score(X, model.labels_):.4f}')
    plt.figure(figsize=(6,3))
    show_silhouette(X=X, fitted_model=model)

## 2.) Image segmentation
<a id='section_2'></a>

### Read the image

In [ ]:
# Read the image
image = cv2.imread('parrot.jpg')
 
# Change the color to RGB (from BGR)
image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# Plot the image
plt.figure(figsize=(6,8))
plt.imshow(image)

### Reshape the image

In [ ]:
# Reshaping the image into a 2D array of pixels and RGB colors
pixel_vals = image.reshape((-1,3))
 
# Convert to float
pixel_vals = np.float32(pixel_vals)
pixel_vals

### Elbow method showing the optimal k

In [ ]:
# Sum of squared distances of samples to their closest cluster center
distortions = []

# Range of k's
K = range(2,10,1)

# Loop to find the optimal k
for k in K:
    kmeanModel = KMeans(n_clusters=k)
    kmeanModel.fit(pixel_vals)
    distortions.append(kmeanModel.inertia_)
    
# Elbow plot
plt.figure(figsize=(5,3))
plt.plot(K, distortions, 'bx-')
plt.xlabel('k')
plt.ylabel('Distortion')
plt.title('The Elbow method showing the optimal k')

plt.show()

### Perform image segmentation

In [ ]:
# Number of clusters
k = 5

# Criteria for the segmentation algorithm to stop running
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.85)
 
# Perform k-means clustering
retval, labels, centers = cv2.kmeans(pixel_vals, 
                                     k, 
                                     None, 
                                     criteria, 
                                     10, 
                                     cv2.KMEANS_RANDOM_CENTERS)

# Print cluster labels
print('Cluster labels:')
print(labels, '\n')

# Print cluster centroids
print(f'Centroids of {k} clusters')
print(centers)

### Change data types and reshape the segmented data for visualization

In [ ]:
# Convert data into 8-bit values
centers = np.uint8(centers)
segmented_data = centers[labels.flatten()]
 
# Reshape data into the original image dimensions
segmented_image = segmented_data.reshape((image.shape))

# Show result
plt.figure(figsize=(6,8))
plt.imshow(segmented_image)

## Task 1d-f: Image Segmentation with Custom Image

For this task, use your own image file (< 500 KB recommended). Place the image in the same directory as this notebook and update the filename in the code below.

In [ ]:
# Task 1d: Read your own image (update filename as needed)
# Example: image_custom = cv2.imread('your_image_name.jpg')
# For demonstration, we'll use the tractor image from the dataset
image_custom = cv2.imread('deutz_d15.jpg')
 
# Change the color to RGB (from BGR)
image_custom = cv2.cvtColor(image_custom, cv2.COLOR_BGR2RGB)

# Plot the image
plt.figure(figsize=(6,8))
plt.imshow(image_custom)
plt.title('Original Custom Image')
plt.axis('off')
plt.show()

In [ ]:
# Reshape the custom image
pixel_vals_custom = image_custom.reshape((-1,3))
pixel_vals_custom = np.float32(pixel_vals_custom)

### Task 1e: Elbow method for custom image

In [ ]:
# Task 1e: Elbow method for custom image
distortions_custom = []
K_custom = range(2,10,1)

for k in K_custom:
    kmeanModel = KMeans(n_clusters=k, random_state=42)
    kmeanModel.fit(pixel_vals_custom)
    distortions_custom.append(kmeanModel.inertia_)
    
# Elbow plot
plt.figure(figsize=(5,3))
plt.plot(K_custom, distortions_custom, 'bx-')
plt.xlabel('k')
plt.ylabel('Distortion')
plt.title('Elbow Method for Custom Image')
plt.show()

print('Based on the elbow chart, the optimal k can be identified where the curve bends significantly.')

### Task 1f: Create 4 different image segmentations with different k values

In [ ]:
# Task 1f: Create 4 different segmentations with k = 2, 4, 6, 8
k_values = [2, 4, 6, 8]
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 0.85)

fig, axes = plt.subplots(2, 2, figsize=(12, 12))
axes = axes.ravel()

for idx, k in enumerate(k_values):
    # Perform k-means clustering
    retval, labels_custom, centers_custom = cv2.kmeans(pixel_vals_custom, 
                                                       k, 
                                                       None, 
                                                       criteria, 
                                                       10, 
                                                       cv2.KMEANS_RANDOM_CENTERS)
    
    # Convert and reshape for visualization
    centers_custom = np.uint8(centers_custom)
    segmented_data_custom = centers_custom[labels_custom.flatten()]
    segmented_image_custom = segmented_data_custom.reshape((image_custom.shape))
    
    # Plot
    axes[idx].imshow(segmented_image_custom)
    axes[idx].set_title(f'k = {k} clusters')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()

## 3.) 3D point cloud segmentation
- For details see: https://towardsdatascience.com/3d-point-cloud-clustering-tutorial-with-k-means-and-python-c870089f3af8
- Data-Viewer: https://app.flyvast.com/flyvast/app/page-snapshot-viewer.html#/444/9b557b91-8f41-16fa-cd2d-3476a1756611
<a id='section_3'></a>

### Import the data (Airport LIDAR point cloud dataset)

In [ ]:
# Import the data
data = "KME_planes.xyz"
x, y, z, illuminance, reflectance, intensity, nb_of_returns = np.loadtxt(data,
                                                                         skiprows=1, 
                                                                         delimiter=';', 
                                                                         unpack=True)
# Array with positions of points
pcd = np.column_stack((x,y,z))

# Mean of z-variable (height above ground)
print(np.mean(z).round(4))

# Subset of the point cloud 
mask = (z > np.mean(z)) & (x <= 320) & (x >= 230)
spatial_query = pcd[mask]
spatial_query.shape

### 3D plot of the point cloud subset

In [ ]:
# 3D plot of the subset
plt.figure(figsize=(5,5))
ax = plt.axes(projection ='3d')
ax.scatter(x[mask], y[mask], z[mask], c=intensity[mask], s=0.5)
plt.show()

### 2D plot of the point cloud subset

In [ ]:
# 2D plot of the subset
plt.figure(figsize=(4,4))
plt.scatter(x[mask], y[mask], c=intensity[mask], s=0.5)
plt.show()

### Elbow method showing the optimal k

In [ ]:
# Sum of squared distances of samples to their closest cluster center
distortions = []

# Range of k's
K = range(2,15,1)

# Loop to find the optimal k
for k in K:
    kmeanModel = KMeans(n_clusters=k)
    kmeanModel.fit(spatial_query)
    distortions.append(kmeanModel.inertia_)
    
# Elbow plot
plt.figure(figsize=(5,3))
plt.plot(K, distortions, 'bx-')
plt.xlabel('k')
plt.ylabel('Distortion')
plt.title('The Elbow Method showing the optimal k')

plt.show()

### Point cloud segmentation based on k-means clustering

In [ ]:
# Define number of clusters
k = 5

# Stack with x,y,z values
X2 = np.column_stack((x[mask], y[mask], z[mask]))

# Perform k-means clustering
kmeans_aerpl = KMeans(n_clusters=k, random_state=42).fit(X2)

### 2D plot of the segmented point cloud

In [ ]:
# 2D plot of image segmentation
plt.figure(figsize=(4,4))
plt.scatter(x[mask], y[mask], c=kmeans_aerpl.labels_, s = 0.1)
plt.show()

### 3D plot of the segmented point cloud

In [ ]:
# 3D plot of image segmentation
plt.figure(figsize=(5,5))
ax = plt.axes(projection ='3d')
ax.scatter(x[mask], y[mask], z[mask], c=kmeans_aerpl.labels_, s=0.1)
plt.show()

### Calculate the Silhouette Score

In [ ]:
print(f'Silhouette Score: {silhouette_score(X2, kmeans_aerpl.labels_):.4f}')

## Task 1h: 3D Point Cloud Segmentation with Different k Values

In [ ]:
# Task 1h: Test different k values for 3D point cloud segmentation
k_values_3d = [3, 5, 7, 9]

fig = plt.figure(figsize=(16, 12))

for idx, k_val in enumerate(k_values_3d, 1):
    # Perform k-means clustering with different k
    kmeans_test = KMeans(n_clusters=k_val, random_state=42).fit(X2)
    
    # 2D plot
    ax1 = fig.add_subplot(4, 2, idx*2-1)
    ax1.scatter(x[mask], y[mask], c=kmeans_test.labels_, s=0.5)
    ax1.set_title(f'2D View (k={k_val})')
    ax1.set_xlabel('x')
    ax1.set_ylabel('y')
    
    # 3D plot
    ax2 = fig.add_subplot(4, 2, idx*2, projection='3d')
    ax2.scatter(x[mask], y[mask], z[mask], c=kmeans_test.labels_, s=0.5)
    ax2.set_title(f'3D View (k={k_val})')
    ax2.set_xlabel('x')
    ax2.set_ylabel('y')
    ax2.set_zlabel('z')

plt.tight_layout()
plt.show()

### Explanation (Task 1h):
When changing the value of k for the 3D point cloud segmentation, we observe the following:

- **Lower k values (e.g., k=3)**: The point cloud is divided into fewer, larger clusters. This provides a coarse segmentation where broad regions or objects are grouped together. This might be suitable for identifying major structures (e.g., different planes or large sections of the airport).

- **Higher k values (e.g., k=7, 9)**: The point cloud is divided into more, smaller clusters. This provides a finer segmentation that can identify more detailed features and separate objects that are spatially close but distinct. This allows for more precise identification of individual objects or parts.

The optimal k depends on the application: if we want to identify individual airplanes, we might use k corresponding to the expected number of planes plus background structures. The elbow method suggests k=5 as a reasonable balance, which likely corresponds to the major objects visible in the airport point cloud.

## 4.) Finding clusters in the apartment data
<a id='section_4'></a>

### Import the apartment data

In [ ]:
# Define columns for import
columns = [ 'web-scraper-order',
            'address_raw',
            'rooms',
            'area',
            'luxurious',
            'price',
            'price_per_m2',
            'lat',
            'lon',
            'bfs_number',
            'bfs_name',
            'pop',
            'pop_dens',
            'frg_pct',
            'emp',
            'mean_taxable_income',
            'dist_supermarket']

# Read and select variables
df_orig = pd.read_csv("apartments_data_enriched_cleaned.csv", sep=";", encoding='utf-8')[columns]

# Rename variable 'web-scraper-order' to 'apmt_id'
df_orig = df_orig.rename(columns={'web-scraper-order': 'id'})

# Remove missing values
df = df_orig.dropna()
df.head(5)

# Remove duplicates
df = df.drop_duplicates()

# Remove some 'extreme' values
df = df.loc[(df['price'] >= 1000) & 
            (df['price'] <= 5000)]

print(df.shape)
df.head(5)

### Task 2b: Extended subset of the apartment data frame for k-means clustering

In [ ]:
# Task 2b: Define an extended subset with additional numerical variables
X3 = df[['rooms',
         'area',
         'price_per_m2',
         'lat',
         'lon',
         'pop_dens',
         'mean_taxable_income']]

print(f'Shape of X3: {X3.shape}')
X3.head()

### Task 2c: Elbow method for extended data frame showing the optimal k

In [ ]:
# Sum of squared distances of samples to their closest cluster center
distortions = []

# Range of k's
K = range(1,15)

# Loop to find the optimal k
for k in K:
    kmeanModel = KMeans(n_clusters=k)
    kmeanModel.fit(X3)
    distortions.append(kmeanModel.inertia_)
    
# Elbow plot
plt.figure(figsize=(5,3))
plt.plot(K, distortions, 'bx-')
plt.xlabel('k')
plt.ylabel('Distortion')
plt.title('The Elbow Method showing the optimal k')

plt.show()

### Task 2d: Perform k-means clustering on the extended apartment data

In [ ]:
# Number of clusters
k = 5

# Perform k-means clustering
kmeans_apmts = KMeans(n_clusters=k, random_state=42).fit(X3)

# Add the clusters to data frame
X3['cluster'] = kmeans_apmts.predict(X3)

# Show number of apartments per cluster
X3['cluster'].value_counts().sort_values(ascending=False)

### Plot the apartment clusters

In [ ]:
fig = plt.figure(figsize=(6,5))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(df['rooms'], df['area'], df['price_per_m2'], c=kmeans_apmts.labels_, s=4)

# Set title and axes labels
ax.set_title('Apartment data clusters', fontsize=12)
ax.set_xlabel('rooms', fontsize=10)
ax.set_ylabel('area', fontsize=10)
ax.set_zlabel('price_per_m2', fontsize=10)

# Set axes range
ax.set_xlim([0,9])
ax.set_ylim([0,150])
ax.set_zlim([20,100])

plt.show()

## Task 2e: Display k-means model attributes

In [ ]:
# Task 2e: Derive attribute values from kmeans_apmts
print('labels_:')
print(kmeans_apmts.labels_, '\n')

print('inertia_:')
print(kmeans_apmts.inertia_, '\n')

print('cluster_centers_:')
print(kmeans_apmts.cluster_centers_, '\n')

print('feature_names_in_:')
print(kmeans_apmts.feature_names_in_)

### Task 2f: Explanation of k-means attributes

Based on the [sklearn.cluster.KMeans documentation](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html):

**1. labels_**: 
- Array of shape (n_samples,)
- Contains the cluster label for each sample (data point)
- Values range from 0 to (k-1), where k is the number of clusters
- Example: If labels_[0] = 2, it means the first apartment belongs to cluster 2

**2. inertia_**:
- Floating point number
- Sum of squared distances of samples to their closest cluster center
- Also known as Within-Cluster Sum of Squares (WCSS)
- Lower values indicate tighter, more compact clusters
- Used in the elbow method to determine optimal k

**3. cluster_centers_**:
- Array of shape (n_clusters, n_features)
- Coordinates of cluster centers (centroids)
- Each row represents one cluster's center in the feature space
- For our apartment data, each centroid shows the average values of rooms, area, price_per_m2, lat, lon, pop_dens, and mean_taxable_income for that cluster

**4. feature_names_in_**:
- Array of feature names
- Contains the names of the features used in the model
- In our case: ['rooms', 'area', 'price_per_m2', 'lat', 'lon', 'pop_dens', 'mean_taxable_income']
- Helps interpret the cluster_centers_ by showing which column corresponds to which feature

### Task 2g: Calculate the Silhouette Score and compare with elbow method

In [ ]:
# Task 2g: Calculate Silhouette Scores for different k values
silhouette_scores = []
K_range = range(2, 11)

for k in K_range:
    kmeans_temp = KMeans(n_clusters=k, random_state=42).fit(X3)
    score = silhouette_score(X3, kmeans_temp.labels_)
    silhouette_scores.append(score)
    print(f'k={k}, Silhouette Score: {score:.4f}')

# Plot Silhouette Scores
plt.figure(figsize=(6,4))
plt.plot(K_range, silhouette_scores, 'ro-')
plt.xlabel('Number of clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score vs Number of Clusters')
plt.grid(True, alpha=0.3)
plt.show()

# Find the k with highest Silhouette Score
best_k_silhouette = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f'\nHighest Silhouette Score: {max(silhouette_scores):.4f} at k={best_k_silhouette}')

### Comparison: Elbow Method vs Silhouette Score (Task 2g)

The elbow method and Silhouette Score may suggest different optimal values for k:

- **Elbow Method**: Identifies the point where adding more clusters provides diminishing returns in reducing within-cluster variance (inertia). The "elbow" represents a balance between model complexity and performance.

- **Silhouette Score**: Measures how similar each point is to its own cluster compared to other clusters. Values range from -1 to 1, where higher values indicate better-defined clusters. The k with the highest Silhouette Score indicates the best cluster separation.

**Analysis**:
If the elbow method and Silhouette Score suggest different k values, this indicates a trade-off:
- The elbow method may suggest a higher k for better within-cluster compactness
- The Silhouette Score may prefer a lower k for better cluster separation

The choice depends on the application: if we want well-separated, distinct apartment groups, we should favor the Silhouette Score. If we want to minimize within-cluster variance and accept some cluster overlap, we should favor the elbow method. In practice, domain knowledge about apartment characteristics should guide the final decision.

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')